# obqac: one optimisation step per sample

Python + CasADi implementation of Deniz and Auat Cheein, *A Single-Step Optimization-Based Kinematic Controller for Real-Time Path Tracking of Legged Robots*, IEEE Trans. Industrial Informatics 22(2), 2026. [doi:10.1109/TII.2025.3609176](https://doi.org/10.1109/TII.2025.3609176)

A quadruped's built-in gait controller takes a forward speed, a lateral speed and a yaw rate. Seen from above the robot is then `q_{k+1} = q_k + F(q_k) u_k Ts` with `F` a rotation, so the control that reaches the next reference pose in one sample is `F^T Delta / Ts`. The controller keeps that idea and adds the velocity and acceleration limits back as a four-variable QP. This notebook runs it against an NMPC with a 20-sample horizon.

In [ ]:
import os, sys
if 'google.colab' in sys.modules and not os.path.isdir('obqac'):
    !git clone -q https://github.com/nahueldeniz/obqac
    %cd obqac
    !pip -q install casadi
for p in (os.getcwd(), os.path.dirname(os.getcwd())):   # repo root, or notebooks/ inside it
    if os.path.isdir(os.path.join(p, 'obqac')):
        sys.path.insert(0, p)
import numpy as np
import matplotlib.pyplot as plt
from obqac import OBQAC, NMPC, LTVLQR, LeggedKinematics, lemniscate, simulate, aggregate
from obqac.viz import plot_runs

## One trial on the lemniscate of Eq. (21)

Reference speed 0.75 m/s, `Ts` = 0.05 s, robot starting 3 m from the path with a random heading, velocity box `[-5pi/4, 5pi/4] x [-1.25, 1.25]^3`, accelerations limited to `pi/2` rad/s² and 0.25 m/s².

In [ ]:
model = LeggedKinematics(ts=0.05)
path = lemniscate()
runs = [simulate(OBQAC(model), path, model, seed=0),
        simulate(NMPC(model, horizon=20, name='NMPC N=20'), path, model, seed=0),
        simulate(LTVLQR(model), path, model, seed=0)]
for r in runs:
    print(f"{r.name:10s} deviation {r.deviation:.4f}  effort {r.effort:.4f}  RTF {r.rtf:.4f}  final error {r.error[-1]:.3f}")
plot_runs(runs, path, 'lemniscate, one trial');

The single-step controller and the 20-step NMPC end up with a similar deviation; the QP costs about a hundredth of the NMPC's time. The LQR tracks best but its commands jump by up to a radian per sample: it does not know about the acceleration limits.

## The error gain

Eq. (10): `e_{k+1} = (1 - delta_e) e_k`. The paper's code uses `delta_e = 1 - 0.9802`. Larger gains reach the path sooner but saturate the actuators; `delta_e = 1` (Eq. 7 taken literally) does not converge under these acceleration limits, see `NOTES.md`.

In [ ]:
plt.figure(figsize=(8, 4))
for de in (0.01, 0.0198, 0.05, 0.2, 1.0):
    r = simulate(OBQAC(model, delta_e=de), path, model, seed=0)
    plt.plot(r.error, label=f'delta_e = {de}')
plt.xlabel('time step'); plt.ylabel('deviation e_k'); plt.legend(); plt.grid(alpha=.3);

## Five trials, paired seeds

The paper's metrics of Eq. (22) averaged over five random initial poses; every controller gets the same five.

In [ ]:
summary = {}
for make in (lambda: OBQAC(model), lambda: NMPC(model, horizon=20, name='NMPC N=20'), lambda: NMPC(model, horizon=1, name='NMPC N=1')):
    rs = [simulate(make(), path, model, seed=i) for i in range(5)]
    s = aggregate(rs); summary[s['controller']] = s
    print(f"{s['controller']:10s} deviation {s['deviation']:.4f} +- {s['deviation_std']:.4f}   effort {s['effort']:.4f}   RTF {s['rtf']:.4f} (max {s['rtf_max']:.3f})")

## Try it

Change the path (`circle()`, `rectangle()`), the speed, the limits (`LeggedKinematics(u_max=..., a_max=...)`) or the weight `W`. `OBQAC(model, solver='closed_form')` is the same controller without a solver at all: for a diagonal `W` the QP is a component-wise clip.